# DuoFusion: do HemoPIC+ maps help nnU-Net predict the final infarct? (ISLES'24)

Two nnU-Net v2 models with the same 5 patient folds:
* **Dataset500 vendor**: NCCT + vendor CBF, CBV, MTT, Tmax (5 channels)
* **Dataset501 vendor + HemoPIC+**: the same + HemoPIC+ CBF, CBV, MTT, delay (9 channels)

Input: `Dataset501_ISLES24Twin.zip` (made by `scripts/export_nnunet.py`) in Google Drive.
Results and checkpoints go to `MyDrive/duofusion/nnunet/`, so a disconnected run resumes by re-running all cells.
Runtime: GPU (H100 / RTX 6000), High-RAM. Pilot: 100 epochs per fold; the two data sets train in parallel.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install nnunetv2
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil, subprocess, time
BASE = '/content/drive/MyDrive/duofusion/nnunet'
os.makedirs(BASE, exist_ok=True)
EPOCHS = 100                      # pilot; 250 for the final runs
TRAINER = f'nnUNetTrainer_{EPOCHS}epochs'
zips = glob.glob('/content/drive/MyDrive/**/Dataset501_ISLES24Twin*.zip', recursive=True)
print(zips)
assert zips, 'upload Dataset501_ISLES24Twin.zip to Google Drive first'

In [ ]:
RAW, PRE = '/content/nnUNet_raw', '/content/nnUNet_preprocessed'
RES = f'{BASE}/nnUNet_results'
for d in (RAW, PRE, RES): os.makedirs(d, exist_ok=True)
os.environ.update(nnUNet_raw=RAW, nnUNet_preprocessed=PRE, nnUNet_results=RES)
if not os.path.exists(f'{RAW}/Dataset501_ISLES24Twin/dataset.json'):
    shutil.copy(zips[0], '/content/ds.zip')
    subprocess.run(['unzip', '-q', '-o', '/content/ds.zip', '-d', RAW], check=True)
src = f'{RAW}/Dataset501_ISLES24Twin'
# vendor-only data set: channels 0-4 of the same cases
dst = f'{RAW}/Dataset500_ISLES24Vendor'
os.makedirs(f'{dst}/imagesTr', exist_ok=True)
shutil.copytree(f'{src}/labelsTr', f'{dst}/labelsTr', dirs_exist_ok=True)
for f in glob.glob(f'{src}/imagesTr/*_000[0-4].nii.gz'):
    t = f'{dst}/imagesTr/' + os.path.basename(f)
    if not os.path.exists(t): os.symlink(f, t)
ds = json.load(open(f'{src}/dataset.json'))
dv = dict(ds, channel_names={k: v for k, v in ds['channel_names'].items() if int(k) < 5},
          channel_descriptions={k: v for k, v in ds['channel_descriptions'].items() if int(k) < 5})
json.dump(dv, open(f'{dst}/dataset.json', 'w'), indent=1)
print(ds['numTraining'], 'cases;', len(glob.glob(f'{src}/imagesTr/*')), 'images')

In [ ]:
for did in (500, 501):
    if not glob.glob(f'{PRE}/Dataset{did}_*/nnUNetPlans.json'):
        subprocess.run(['nnUNetv2_plan_and_preprocess', '-d', str(did), '-c', '3d_fullres', '--verify_dataset_integrity', '-np', '8'], check=True)
    name = os.path.basename(glob.glob(f'{RAW}/Dataset{did}_*')[0])
    shutil.copy(f'{src}/splits_final.json', f'{PRE}/{name}/splits_final.json')   # the duofusion patient folds
print(open(glob.glob(f'{PRE}/Dataset501_*/nnUNetPlans.json')[0]).read()[:1500])

In [ ]:
def run_all_folds(did, log):
    # one background process that trains the 5 folds of one data set in turn (resumes if interrupted)
    script = f'''
set -e
for f in 0 1 2 3 4; do
  d=$(ls -d {RES}/Dataset{did}_*/{TRAINER}__nnUNetPlans__3d_fullres/fold_$f 2>/dev/null || true)
  if [ -n "$d" ] && [ -f "$d/checkpoint_final.pth" ]; then continue; fi
  if [ -n "$d" ] && [ -f "$d/checkpoint_latest.pth" ]; then C=--c; else C=; fi
  nnUNet_n_proc_DA=6 nnUNetv2_train {did} 3d_fullres $f -tr {TRAINER} --npz $C
done
echo ALL_DONE
'''
    return subprocess.Popen(['bash', '-c', script], stdout=open(log, 'a'), stderr=subprocess.STDOUT)

procs = {did: run_all_folds(did, f'{BASE}/train_{did}.log') for did in (500, 501)}
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for did in procs:
        tail = subprocess.run(['tail', '-n', '2', f'{BASE}/train_{did}.log'], capture_output=True, text=True).stdout.strip()
        print(time.strftime('%H:%M'), did, tail[-160:])
print({did: p.returncode for did, p in procs.items()})

In [ ]:
import numpy as np, nibabel as nib, csv
from scipy.stats import wilcoxon
def dice(a, b):
    s = a.sum() + b.sum()
    return 2 * (a & b).sum() / s if s else np.nan
rows = {}
for did in (500, 501):
    for f in glob.glob(f'{RES}/Dataset{did}_*/{TRAINER}__nnUNetPlans__3d_fullres/fold_*/validation/*.nii.gz'):
        cid = os.path.basename(f)[:-7]
        img = nib.load(f); pred = np.asarray(img.dataobj) > 0
        gt = np.asarray(nib.load(f'{src}/labelsTr/{cid}.nii.gz').dataobj) > 0
        ml = float(np.prod(img.header.get_zooms()[:3])) / 1000
        rows.setdefault(cid, {})[did] = (dice(pred, gt), abs(pred.sum() - gt.sum()) * ml, gt.sum() * ml)
common = [c for c in rows if 500 in rows[c] and 501 in rows[c]]
d0 = np.array([rows[c][500][0] for c in common]); d1 = np.array([rows[c][501][0] for c in common])
v0 = np.array([rows[c][500][1] for c in common]); v1 = np.array([rows[c][501][1] for c in common])
ok = np.isfinite(d0) & np.isfinite(d1)
rng = np.random.default_rng(0); diff = (d1 - d0)[ok]
boot = diff[rng.integers(0, diff.size, (5000, diff.size))].mean(1)
res = {'n': int(ok.sum()), 'dice_vendor': float(np.nanmean(d0)), 'dice_vendor_twin': float(np.nanmean(d1)),
       'dice_gain': float(diff.mean()), 'dice_gain_ci95': [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
       'wins': int((diff > 0).sum()), 'wilcoxon_p': float(wilcoxon(diff).pvalue),
       'abs_vol_err_ml_vendor': float(np.median(v0)), 'abs_vol_err_ml_vendor_twin': float(np.median(v1))}
print(json.dumps(res, indent=1))
json.dump(res, open(f'{BASE}/summary_{EPOCHS}ep.json', 'w'), indent=1)
with open(f'{BASE}/per_case_{EPOCHS}ep.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(['case', 'dice_vendor', 'dice_vendor_twin', 'abs_vol_err_vendor', 'abs_vol_err_vendor_twin', 'truth_ml'])
    for c in common: w.writerow([c, rows[c][500][0], rows[c][501][0], rows[c][500][1], rows[c][501][1], rows[c][500][2]])